In [71]:
from wgpu import BufferUsage, ShaderStage, BufferBindingType
import wgpu
import numpy as np
import random as rd

1. Les données sur le CPU, avec une version CPU de référence.


In [72]:
def create_initial_data(size:(int, int), starting_population:int):
    data0 = np.zeros(size, dtype=np.uint32)
    data1 = np.zeros(size, dtype=np.uint32)
    for i in range(starting_population):
        x = rd.randint(0, size[0]-1)
        y = rd.randint(0, size[1]-1)
        if x == 0 or x == size[0]-1 or y == 0 or y == size[1]-1:
            pass
        else:
            data0[x][y] = 1
    return data0,data1
data0, data1= create_initial_data((10, 10), 50)

2. Le canal avec le GPU.

In [73]:
adapter = wgpu.gpu.request_adapter_sync(power_preference="high-performance")
device = adapter.request_device_sync()

3. Les buffers, y compris le petit buffer de réglages.

In [74]:
buffer_0 = device.create_buffer_with_data(data=data0, usage=BufferUsage.STORAGE | BufferUsage.COPY_SRC)
buffer_1 = device.create_buffer_with_data(data=data1, usage=BufferUsage.STORAGE | BufferUsage.COPY_SRC)

4. Le contrat des emplacements.

In [75]:
bind_group_layout = device.create_bind_group_layout(entries=[
    {"binding": 0, "visibility": ShaderStage.COMPUTE, "buffer": {"type": BufferBindingType.storage}},
    {"binding": 1, "visibility": ShaderStage.COMPUTE, "buffer": {"type": BufferBindingType.storage}},
])

5. Les bind groups, un par sens du ping-pong.

In [76]:
bind_group = device.create_bind_group(layout=bind_group_layout, entries=[
    {"binding": 0, "resource": {"buffer": buffer_0, "offset": 0, "size": buffer_0.size}},
    {"binding": 1, "resource": {"buffer": buffer_1, "offset": 0, "size": buffer_1.size}},
])
bind_group_reverse = device.create_bind_group(layout=bind_group_layout, entries=[
    {"binding": 0, "resource": {"buffer": buffer_1, "offset": 0, "size": buffer_1.size}},
    {"binding": 1, "resource": {"buffer": buffer_0, "offset": 0, "size": buffer_0.size}},
])
bind_groups = [bind_group, bind_group_reverse]
written_buffers = [buffer_1, buffer_0]

6. Le shader.

In [77]:
HEIGHT, WIDTH = data0.shape

shader_header = f"const WIDTH: u32 = {WIDTH}u;\nconst HEIGHT: u32 = {HEIGHT}u;\n"
shader_body = """
@group(0) @binding(0) var<storage, read_write> source: array<u32>;
@group(0) @binding(1) var<storage, read_write> cible: array<u32>;

/* @brief Converts a (row, col) position into an index in the flat cell buffer.
   @param row Row of the cell
   @param col Column of the cell */
fn cell_index(row: u32, col: u32) -> u32 {
    return row * WIDTH + col;
}

/* @brief Computes the next Game of Life state of the cell owned by this thread.
   @param id Global invocation id: id.x is the column, id.y is the row */
@compute @workgroup_size(8, 8)
fn main(@builtin(global_invocation_id) id: vec3<u32>) {
    let col = id.x;
    let row = id.y;

    if (col >= WIDTH || row >= HEIGHT) {
        return;
    }

    let me = cell_index(row, col);

    if (col == 0u || row == 0u || col == WIDTH - 1u || row == HEIGHT - 1u) {
        cible[me] = 0u;
        return;
    }

    var alive_neighbors = 0u;
    for (var dr = 0u; dr < 3u; dr = dr + 1u) {
        for (var dc = 0u; dc < 3u; dc = dc + 1u) {
            if (dr == 1u && dc == 1u) {
                continue;
            }
            alive_neighbors += source[cell_index(row + dr - 1u, col + dc - 1u)];
        }
    }

    let alive = source[me] == 1u;
    var next = 0u;
    if (alive_neighbors == 3u || (alive && alive_neighbors == 2u)) {
        next = 1u;
    }
    cible[me] = next;
}
"""
shader_source = shader_header + shader_body

shader_module = device.create_shader_module(code=shader_source)

7. Le pipeline.

In [78]:
pipeline_layout = device.create_pipeline_layout(bind_group_layouts=[bind_group_layout])
pipeline = device.create_compute_pipeline(
    layout=pipeline_layout,
    compute={"module": shader_module, "entry_point": "main"})

8. lancer une génération et comparer

In [79]:
def run_generation(bind_group, result_buffer):
    encoder = device.create_command_encoder()
    compute_pass = encoder.begin_compute_pass()
    compute_pass.set_pipeline(pipeline)
    compute_pass.set_bind_group(0, bind_group)
    compute_pass.dispatch_workgroups((WIDTH + 7) // 8, (HEIGHT + 7) // 8, 1)
    compute_pass.end()
    device.queue.submit([encoder.finish()])
    raw = device.queue.read_buffer(result_buffer)
    return np.frombuffer(raw, dtype=np.uint32).reshape(HEIGHT, WIDTH).copy()

In [80]:
import time
from IPython.display import clear_output

"""
/* @brief Converts a grid into a printable text block, one character per cell.
   @param grid 2D array of cells, 1 for alive and 0 for dead */
"""
def draw(grid):
    return "\n".join("".join("█" if cell else "·" for cell in row) for row in grid)

generation = 0
while True:
    parity = generation % 2
    grid = run_generation(bind_groups[parity], written_buffers[parity])
    clear_output(wait=True)
    print(f"Génération {generation + 1}")
    print(draw(grid))
    time.sleep(0.5)
    generation += 1

Génération 26
··········
·····█····
·····█····
·····█····
··········
·███······
··········
··········
··········
··········


KeyboardInterrupt: 